# Table 6 — Heterogeneous Effects of Inflation

Replication code for the heterogeneity regressions from:

> Albertazzi, U., 't Hooft, J., & ter Steege, L. (2025).  
> *The causal effect of inflation on financial stability: evidence from history.*  
> ECB Working Paper Series No. 3108.

---

## Methodology

Table 6 extends Equation 4 by interacting instrumented inflation
$\widehat{\Delta\pi}_{i,t-1}$ with country-level heterogeneity variables
to test the redistribution channel — the mechanism through which inflation
erodes real household income and raises crisis risk.

| Column | Heterogeneity variable | Channel tested |
|---|---|---|
| 1 | Wage growth | Do wages keep pace with inflation? |
| 2 | Mortgage, corporate and public debt ratios | Balance sheet vulnerability |
| 3 | CB financial independence | Credibility anchor for inflation expectations |

All columns share the same first stage (Equation 3). Controls are contemporaneous
and four lags of short-term interest rate changes and real GDP growth.
Country fixed effects throughout. Sample: 1975–2020.

## 1. Imports

In [46]:
import pandas as pd
import numpy as np
from utils import (
    melt, winsor,
    display, display_w_std_errors,
    run_first_stage, run_heterogeneity_regression,
)
from data import (
    load_jst, load_romelli, load_oil_shocks,
    build_oil_shocks_panel, build_base_panel,
)

## 2. Data

See `data.py` for full source details and download instructions.

In [47]:
raw_df     = load_jst()
oil_shocks = load_oil_shocks()

# Merge Romelli CBI index before pivoting
raw_df = raw_df.merge(load_romelli(), how='left', on=['year', 'country'])

def pivot(v): return raw_df.pivot(index='year', columns='country', values=v)

## 3. Variable Construction

In [48]:
# ── Core macro series ──────────────────────────────────────────────────────────
cpi_df    = pivot('cpi')
gdp_df    = pivot('rgdpbarro')
stir_df   = pivot('stir')
crisis_df = pivot('crisisJST')

# ── Instrument ────────────────────────────────────────────────────────────────
oil_shocks_df = build_oil_shocks_panel(cpi_df, oil_shocks)

# ── Heterogeneity variables ────────────────────────────────────────────────────
# Column 1: wage growth (annual, winsorised) — proxy for real income recovery speed
wages_df = winsor(pivot('wage').pct_change(fill_method=None) * 100)

# Column 2: debt ratios (winsorised)
mortgage_to_gdp_df       = winsor(pivot('tmort') / pivot('gdp'))
corporate_debt_to_gdp_df = winsor(pivot('bdebt') / pivot('gdp'))
public_debt_df           = pivot('debtgdp')

# Column 3: CB financial independence, demeaned by the cross-sectional median.
# A positive value means the CB is more independent than peers in that year.
cb_fin_indep_df          = pivot('cbie_finindep')
cb_fin_indep_relative_df = cb_fin_indep_df.sub(cb_fin_indep_df.median(axis=1), axis=0)

## 4. Regression Panel

Built once and shared across all three heterogeneity regressions.
Heterogeneity levels are added here; their interactions with instrumented
inflation follow in Section 6 once fitted values are available.

In [49]:
panel_df, covariates = build_base_panel(
    raw_df, crisis_df, cpi_df, gdp_df, stir_df, oil_shocks_df)

# Add heterogeneity levels, lagged 2 years to reduce reverse-causality risk
panel_df['wages']          = melt(wages_df.shift(2))
panel_df['mortgages']      = melt(mortgage_to_gdp_df.shift(2))
panel_df['corporate_debt'] = melt(corporate_debt_to_gdp_df.shift(2))
panel_df['public_debt']    = melt(public_debt_df.shift(2))
panel_df['cb_fin_indep']   = melt(cb_fin_indep_relative_df.shift(2))

## 5. First Stage — Equation 3

Common across all three columns of Table 6.

In [50]:
first_stage_fit, fitted_df = run_first_stage(panel_df, covariates)

display(first_stage_fit).iloc[:2]  # oil_shock coefficient

,results
const,-0.493***
oil_shock,-0.060***


## 6. Interaction Terms

With fitted values in hand, we construct instrumented inflation
($\widehat{\Delta\pi}_{i,t-1}$) and its interactions with each heterogeneity
variable. A negative interaction means the variable *attenuates* the baseline
inflation-crisis link; positive means *amplification*.

In [51]:
infl_hat_1 = fitted_df.shift(1)  # instrumented inflation, lagged one year
panel_df['delta_inflation_hat'] = melt(infl_hat_1)

# Column 1: wage growth interaction
panel_df['infl_hat X wages']          = melt(infl_hat_1 * wages_df.shift(2))

# Column 2: debt interactions
panel_df['infl_hat X mortgages']      = melt(infl_hat_1 * mortgage_to_gdp_df.shift(2))
panel_df['infl_hat X corporate_debt'] = melt(infl_hat_1 * corporate_debt_to_gdp_df.shift(2))
panel_df['infl_hat X public_debt']    = melt(infl_hat_1 * public_debt_df.shift(2))

# Column 3: CB financial independence interaction
panel_df['infl_hat X cb_fin_indep']   = melt(infl_hat_1 * cb_fin_indep_relative_df.shift(2))

## 7. Table 6 — Heterogeneity Regressions

Each cell calls `run_heterogeneity_regression` from `utils.py` with the
relevant interaction and level variables. Controls and NaN handling are
managed by the shared function.

### Column 1: Wage Growth

Countries with lower prior wage growth should see a stronger inflation-crisis link,
since households have less capacity to absorb real income erosion.
Expected sign on the interaction: **negative**.

In [52]:
run_heterogeneity_regression(panel_df, covariates, ['infl_hat X wages', 'wages'])

,results_w_errors
const,0.055*** (0.011)
delta_inflation_hat,0.052*** (0.017)
infl_hat X wages,-0.001** (0.001)
wages,-0.001 (0.001)


### Column 2: Debt

The redistribution channel predicts amplification through *household* debt (mortgages)
but not necessarily through corporate or public debt.
Expected sign on the mortgage interaction: **positive**.

In [53]:
run_heterogeneity_regression(panel_df, covariates, [
    'infl_hat X mortgages',      'mortgages',
    'infl_hat X corporate_debt', 'corporate_debt',
    'infl_hat X public_debt',    'public_debt',
])

,results_w_errors
const,-0.004 (0.034)
delta_inflation_hat,-0.012 (0.025)
infl_hat X mortgages,0.080*** (0.026)
mortgages,0.064 (0.048)
infl_hat X corporate_debt,0.012 (0.008)
corporate_debt,0.043 (0.032)
infl_hat X public_debt,0.020 (0.02)
public_debt,-0.024 (0.02)


### Column 3: Central Bank Financial Independence

Higher CB financial independence should anchor inflation expectations,
dampening second-round effects and reducing the crisis impact of inflationary shocks.
Expected sign on the interaction: **negative**.

In [54]:
run_heterogeneity_regression(panel_df, covariates, ['infl_hat X cb_fin_indep', 'cb_fin_indep'])

,results_w_errors
const,0.054*** (0.012)
delta_inflation_hat,0.044** (0.017)
infl_hat X cb_fin_indep,-0.073* (0.044)
cb_fin_indep,0.021 (0.038)
